In [1]:
# Базовые понятия
# async def — объявляет асинхронную функцию (корутину).
# await — точка «ожидания», где корутина отдаёт управление event loop.
# Event loop — цикл событий, который переключается между корутинами.
# asyncio.run() — запускает event loop и вашу главную корутину.

In [3]:
# Пример 1: Минимальная асинхронная программа
# потому что вы вызываете asyncio.run() внутри уже запущенного event loop — типично 
# для Jupyter/VS Code Notebook, где асинхронный цикл уже активен. Решение для ноутбуков (Jupyter, VS Code)
# Вместо asyncio.run() просто используйте await напрямую в ячейке:

import asyncio

async def main():
    print("Старт")
    await asyncio.sleep(1)  # не блокирует поток
    print("Прошла 1 секунда")

# asyncio.run(main())
# В ноутбуке — просто await, без asyncio.run()
await main()

Старт
Прошла 1 секунда


In [ ]:
# Пример 2: Параллельный запуск нескольких задач
# Если делать await подряд — будет последовательно. Для конкурентного выполнения 
# используют asyncio.gather или asyncio.create_task.

import asyncio

async def say_hello(name: str, delay: int):
    await asyncio.sleep(delay)
    print(f"Привет, {name}!")

async def main():
    await asyncio.gather(
        say_hello("Alice", 2),
        say_hello("Bob", 1),
        say_hello("Charlie", 3),
    )

await main()

# asyncio.run(main())
# Все три корутины стартуют «почти одновременно», и общее время выполнения ≈ 3 секунды, а не 2+1+3.

Привет, Bob!
Привет, Alice!
Привет, Charlie!


In [ ]:
# Пример 3: Асинхронные HTTP-запросы (практический сценарий)
# Для реального I/O (сеть, файлы) используют библиотеки поверх asyncio, например aiohttp.

import asyncio
import aiohttp

async def fetch(url: str, session: aiohttp.ClientSession) -> str:
    async with session.get(url) as resp:
        return await resp.text()

async def main():
    urls = [
        "https://jsonplaceholder.typicode.com/todos/1",
        "https://jsonplaceholder.typicode.com/todos/2",
        "https://jsonplaceholder.typicode.com/todos/3",
    ]

    async with aiohttp.ClientSession() as session:
        tasks = [fetch(u, session) for u in urls]
        results = await asyncio.gather(*tasks)

    for i, text in enumerate(results, 1):
        print(f"Ответ {i}: {text[:100]}...")

# asyncio.run(main())
# Здесь все запросы идут параллельно в одном потоке, что сильно быстрее последовательных requests.get.
await main()

/media/gansior/t128/python_workshops/.py_venv/lib/python3.12/site-packages/aiohttp/compression_utils.py:193: RuntimeWarning: coroutine 'main' was never awaited
  class ZLibCompressor:


Ответ 1: {
  "userId": 1,
  "id": 1,
  "title": "delectus aut autem",
  "completed": false
}...
Ответ 2: {
  "userId": 1,
  "id": 2,
  "title": "quis ut nam facilis et officia qui",
  "completed": false
}...
Ответ 3: {
  "userId": 1,
  "id": 3,
  "title": "fugiat veniam minus",
  "completed": false
}...


In [8]:
# Пример 4: create_task и фоновые задачи
# asyncio.create_task позволяет запустить корутину «в фоне» и работать с ней как с задачей.

import asyncio

async def worker(task_id: int):
    await asyncio.sleep(4)
    print(f"Задача {task_id} завершена")

async def main():
    t1 = asyncio.create_task(worker(1))
    t2 = asyncio.create_task(worker(2))

    # Делаем что-то ещё, пока задачи работают
    await asyncio.sleep(1)
    print("Основная логика пока работает...")

    await t1, t2  # ждём завершения

# asyncio.run(main())

await main()


Основная логика пока работает...
Задача 1 завершена
Задача 2 завершена
